In [ ]:
import os
import warnings
from itertools import combinations
from datetime import datetime

import numpy as np
import pandas as pd
import yfinance as yf
import pytz
import statsmodels.api as sm

from joblib import Parallel, delayed
from statsmodels.tsa.stattools import coint, adfuller, zivot_andrews

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================

TIMEFRAME_OPTIONS = ['1m', '5m', '15m', '30m', '60m', '1d']

# Yahoo interval + approximate period to fetch enough data
TIMEFRAME_MAPPING = {
    '1m':  {'interval': '1m',  'period': '7d'},
    '5m':  {'interval': '5m',  'period': '60d'},
    '15m': {'interval': '15m', 'period': '60d'},
    '30m': {'interval': '30m', 'period': '60d'},
    '60m': {'interval': '60m', 'period': '730d'},
    '1d':  {'interval': '1d',  'period': '5y'},
}

FORMATION_BARS = {
    '1m': 100,
    '5m': 100,
    '15m': 100,
    '30m': 100,
    '60m': 120,
    '1d': 252,
}

TRADING_BARS = {
    '1m': 30,
    '5m': 30,
    '15m': 30,
    '30m': 30,
    '60m': 60,
    '1d': 63,
}

max_holding_period_bars = 500
num_pairs = 20

z_entry_threshold = 2.0
z_exit_threshold = 0.5
stop_loss_threshold = 0.5
risk_per_trade_pct = 0.01
fee_per_share = 0.005
initial_balance = 100000

# Relaxed validation thresholds
MIN_OUT_SAMPLE_SHARPE = -2.0
MAX_BETA_STD = 5.0
MIN_HALF_LIFE = 1

# Forced pair fallback
FORCE_PAIR_THRESHOLD = 3
FORCE_NUM_PAIRS = 5
FORCED_PAIRS_FLAG = "*** FILTER BYPASSED — FORCED PAIR (correlation only) ***"

CANDIDATE_FORCED_PAIRS = [
    ('AAPL', 'MSFT'),
    ('GOOGL', 'META'),
    ('JPM', 'BAC'),
    ('XOM', 'CVX'),
    ('V', 'MA'),
    ('KO', 'PEP'),
    ('AMZN', 'NFLX'),
    ('NVDA', 'INTC'),
    ('PFE', 'MRK'),
    ('ABBV', 'AMGN'),
    ('ADBE', 'CRM'),
    ('QCOM', 'TXN'),
    ('UNH', 'ABT'),
    ('DIS', 'CMCSA'),
    ('WMT', 'COST'),
]

# Universe
EQUITY_SYMBOLS = [
    'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'META', 'TSLA', 'JNJ', 'V',   'JPM',
    'WMT',  'PG',   'MA',   'UNH',   'DIS',  'NVDA', 'HD',  'PYPL', 'BAC',
    'VZ',   'ADBE', 'CMCSA','NFLX',  'XOM',  'INTC', 'T',   'CSCO', 'PFE',
    'KO',   'MRK',  'ABBV', 'PEP',   'ABT',  'CRM',  'ACN', 'MDT',  'COST',
    'WFC',  'TMO',  'DHR',  'AMGN',  'QCOM', 'TXN',  'NEE', 'ORCL', 'UPS',
    'BMY',  'MS',   'LIN',  'CVX'
]

# Optional Yahoo symbols for FX / commodities
YAHOO_EXTRA_SYMBOLS = {
    'EURUSD': 'EURUSD=X',
    'USDJPY': 'JPY=X',
    'GBPUSD': 'GBPUSD=X',
    'XAUUSD': 'GC=F'
}

timezone = pytz.timezone('US/Eastern')

# =============================================================================
# HELPERS
# =============================================================================

def now_eastern():
    return datetime.now(pytz.utc).astimezone(timezone)

def safe_print(msg):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

def convert_symbol_for_yahoo(symbol):
    return YAHOO_EXTRA_SYMBOLS.get(symbol, symbol)

def clean_ohlc_dataframe(df):
    if df is None or df.empty:
        return None

    # Handle possible multi-index columns from yfinance
    if isinstance(df.columns, pd.MultiIndex):
        if 'Close' in df.columns.get_level_values(0):
            close_df = df['Close'].copy()
        else:
            return None
    else:
        if 'Close' in df.columns:
            close_df = df[['Close']].copy()
            close_df.columns = [df.attrs.get('symbol', 'UNKNOWN')]
        else:
            return None

    close_df = close_df.dropna(how='all')
    close_df.index = pd.to_datetime(close_df.index)
    if close_df.index.tz is None:
        close_df.index = close_df.index.tz_localize('UTC').tz_convert('US/Eastern')
    else:
        close_df.index = close_df.index.tz_convert('US/Eastern')
    close_df = close_df.sort_index()
    close_df = close_df[~close_df.index.duplicated(keep='last')]
    return close_df

def download_price_data(symbols, timeframe='1d', include_extra=False):
    cfg = TIMEFRAME_MAPPING[timeframe]
    interval = cfg['interval']
    period = cfg['period']

    if include_extra:
        all_symbols = symbols + [s for s in YAHOO_EXTRA_SYMBOLS.keys() if s not in symbols]
    else:
        all_symbols = symbols[:]

    yahoo_symbols = [convert_symbol_for_yahoo(s) for s in all_symbols]

    safe_print(f"Downloading data: interval={interval}, period={period}")
    raw = yf.download(
        tickers=yahoo_symbols,
        period=period,
        interval=interval,
        auto_adjust=False,
        progress=False,
        group_by='column',
        threads=True
    )

    if raw is None or raw.empty:
        raise ValueError("No data downloaded from Yahoo Finance.")

    if not isinstance(raw.columns, pd.MultiIndex):
        # Single symbol case fallback
        original_symbol = all_symbols[0]
        raw.attrs['symbol'] = original_symbol
        close_df = clean_ohlc_dataframe(raw)
        if close_df is None:
            raise ValueError("Could not extract Close prices.")
        close_df.columns = [original_symbol]
        return close_df

    if 'Close' not in raw.columns.get_level_values(0):
        raise ValueError("Downloaded data does not contain Close prices.")

    close_prices = raw['Close'].copy()
    close_prices.index = pd.to_datetime(close_prices.index)
    if close_prices.index.tz is None:
        close_prices.index = close_prices.index.tz_localize('UTC').tz_convert('US/Eastern')
    else:
        close_prices.index = close_prices.index.tz_convert('US/Eastern')

    # Map Yahoo names back to original names
    reverse_map = {convert_symbol_for_yahoo(s): s for s in all_symbols}
    close_prices.columns = [reverse_map.get(col, col) for col in close_prices.columns]
    close_prices = close_prices.sort_index()
    close_prices = close_prices[~close_prices.index.duplicated(keep='last')]

    # Keep only columns with enough non-null values
    min_non_na = max(50, FORMATION_BARS[timeframe] + TRADING_BARS[timeframe])
    keep_cols = [c for c in close_prices.columns if close_prices[c].notna().sum() >= min_non_na]
    close_prices = close_prices[keep_cols]

    safe_print(f"Downloaded close prices for {len(close_prices.columns)} symbols.")
    return close_prices

# =============================================================================
# PAIR ANALYSIS
# =============================================================================

def build_forced_pair(stock1, stock2, prices):
    s1 = prices[stock1].dropna()
    s2 = prices[stock2].dropna()
    common = s1.index.intersection(s2.index)
    s1, s2 = s1.loc[common], s2.loc[common]

    if len(s1) < 10:
        return None

    try:
        X = sm.add_constant(s2)
        beta = sm.OLS(s1, X).fit().params.iloc[1]
    except Exception:
        return None

    spread = s1 - beta * s2
    spread_mean = spread.mean()
    spread_std = spread.std()

    if spread_std == 0 or np.isnan(spread_std):
        return None

    try:
        lag = spread.shift(1).dropna()
        diff = (spread - spread.shift(1)).dropna()
        idx = lag.index.intersection(diff.index)
        b = np.polyfit(lag.loc[idx].values, diff.loc[idx].values, 1)[0]
        half_life = -np.log(2) / b if b < 0 else 50.0
    except Exception:
        half_life = 50.0

    return {
        'stock1': stock1,
        'stock2': stock2,
        'beta': beta,
        'half_life': half_life,
        'eg_p_value': np.nan,
        'adf_stat': np.nan,
        'adf_p': np.nan,
        'za_stat': np.nan,
        'za_p': np.nan,
        'out_sample_sr': 0.0,
        'beta_std': np.nan,
        'spread_mean': spread_mean,
        'spread_std': spread_std,
        'forced': True,
    }

def analyze_pairs(window_prices, num_pairs=20):
    prices = window_prices.dropna(axis=1, how='all').copy()

    # Drop columns with too many NaNs
    good_cols = [c for c in prices.columns if prices[c].notna().sum() >= 30]
    prices = prices[good_cols]

    valid_tickers = prices.columns.tolist()
    ticker_pairs = list(combinations(valid_tickers, 2))
    safe_print(f"Analyzing {len(ticker_pairs)} candidate pairs from {len(valid_tickers)} symbols.")

    required_data_points = 30

    def analyze_pair(pair):
        stock1, stock2 = pair
        s1 = prices[stock1].dropna()
        s2 = prices[stock2].dropna()

        common_dates = s1.index.intersection(s2.index)
        s1 = s1.loc[common_dates]
        s2 = s2.loc[common_dates]

        if len(s1) < required_data_points:
            return None
        if s1.nunique() < 2 or s2.nunique() < 2:
            return None

        try:
            eg_stat, eg_p_value, _ = coint(s1, s2)
        except Exception:
            return None

        if eg_p_value > 0.10:
            return None

        split = int(len(s1) * 0.7)
        s1_train = s1.iloc[:split]
        s1_test = s1.iloc[split:]
        s2_train = s2.iloc[:split]
        s2_test = s2.iloc[split:]

        if len(s1_train) < 20 or len(s1_test) < 10:
            return None

        try:
            X_train = sm.add_constant(s2_train)
            model = sm.OLS(s1_train, X_train).fit()
            beta = model.params.iloc[1]
        except Exception:
            return None

        spread_train = s1_train - beta * s2_train
        spread_test = s1_test - beta * s2_test

        try:
            adf_stat, adf_p, *_ = adfuller(spread_train, autolag='AIC')
        except Exception:
            return None

        if adf_p > 0.10:
            return None

        za_stat, za_p = np.nan, 1.0
        if len(spread_train) >= 50:
            try:
                za_stat, za_p, *_ = zivot_andrews(
                    spread_train, trim=0.15, maxlag=None, regression='c', autolag='AIC'
                )
            except Exception:
                za_p = 1.0

        if len(spread_train) >= 50 and za_p > 0.15:
            return None

        spread_lag = spread_train.shift(1).dropna()
        spread_diff = (spread_train - spread_train.shift(1)).dropna()
        idx = spread_lag.index.intersection(spread_diff.index)
        spread_lag = spread_lag.loc[idx]
        spread_diff = spread_diff.loc[idx]

        if len(spread_lag) < 10:
            return None

        try:
            beta_hr = np.polyfit(spread_lag.values, spread_diff.values, 1)[0]
            half_life = -np.log(2) / beta_hr if beta_hr < 0 else np.nan
        except Exception:
            return None

        max_half_life = len(s1_train)
        if np.isnan(half_life) or not (MIN_HALF_LIFE <= half_life <= max_half_life):
            return None

        spread_mean = spread_train.mean()
        spread_std = spread_train.std()

        if spread_std == 0 or np.isnan(spread_std):
            return None

        z_test = (spread_test - spread_mean) / spread_std
        if len(z_test) < 3:
            return None

        signal = -np.sign(z_test.values[:-1])
        raw_ret = signal * np.diff(z_test.values)
        sig_chg = np.diff(np.concatenate([[0], signal])) != 0
        costs = sig_chg[:-1] * fee_per_share * 2 if len(sig_chg) > 1 else np.array([])
        ret_net = raw_ret - costs if len(costs) == len(raw_ret) else raw_ret

        if len(ret_net) < 5 or np.std(ret_net) == 0:
            return None

        out_sample_sr = np.mean(ret_net) / np.std(ret_net)
        if out_sample_sr < MIN_OUT_SAMPLE_SHARPE:
            return None

        rolling_window = max(5, int(len(s1_train) * 0.25))
        rolling_betas = []
        for i in range(rolling_window, len(s1_train) + 1):
            try:
                Xr = sm.add_constant(s2_train.iloc[i - rolling_window:i])
                yr = s1_train.iloc[i - rolling_window:i]
                rolling_betas.append(sm.OLS(yr, Xr).fit().params.iloc[1])
            except Exception:
                continue

        if len(rolling_betas) < 3:
            return None

        beta_std = np.std(rolling_betas)
        if beta_std > MAX_BETA_STD:
            return None

        return {
            'stock1': stock1,
            'stock2': stock2,
            'beta': beta,
            'half_life': half_life,
            'eg_p_value': eg_p_value,
            'adf_stat': adf_stat,
            'adf_p': adf_p,
            'za_stat': za_stat,
            'za_p': za_p,
            'out_sample_sr': out_sample_sr,
            'beta_std': beta_std,
            'spread_mean': spread_mean,
            'spread_std': spread_std,
            'forced': False,
        }

    results = Parallel(n_jobs=-1, prefer='threads')(
        delayed(analyze_pair)(pair) for pair in ticker_pairs
    )
    pairs_info = [r for r in results if r is not None]
    pairs_info_sorted = sorted(pairs_info, key=lambda x: x['out_sample_sr'], reverse=True)
    selected = pairs_info_sorted[:num_pairs]

    safe_print(f"Full-filter selected {len(selected)} pairs.")

    if len(selected) < FORCE_PAIR_THRESHOLD:
        safe_print(
            f"WARNING: Only {len(selected)} pairs survived full filter. "
            f"Activating forced-pair mode."
        )
        existing_pairs = {(p['stock1'], p['stock2']) for p in selected}
        forced_count = 0

        for s1, s2 in CANDIDATE_FORCED_PAIRS:
            if forced_count >= FORCE_NUM_PAIRS:
                break
            if (s1, s2) in existing_pairs or (s2, s1) in existing_pairs:
                continue
            if s1 not in prices.columns or s2 not in prices.columns:
                continue

            fp = build_forced_pair(s1, s2, prices)
            if fp is None:
                continue

            selected.append(fp)
            existing_pairs.add((s1, s2))
            forced_count += 1
            safe_print(
                f"{FORCED_PAIRS_FLAG} Added forced pair {s1}/{s2} "
                f"beta={fp['beta']:.4f} HL={fp['half_life']:.1f}"
            )

    for p in selected[:5]:
        tag = " [FORCED]" if p.get('forced') else ""
        safe_print(
            f"{p['stock1']}/{p['stock2']}{tag} | "
            f"EG_p={p['eg_p_value'] if not np.isnan(p['eg_p_value']) else 'N/A'} | "
            f"ADF_p={p['adf_p'] if not np.isnan(p['adf_p']) else 'N/A'} | "
            f"HL={p['half_life']:.1f} | "
            f"OOS_SR={p['out_sample_sr']:.3f}"
        )

    return selected

# =============================================================================
# BACKTEST ENGINE
# =============================================================================

def open_position(position_list, timestamp, pair, price1, price2, equity):
    stock1 = pair['stock1']
    stock2 = pair['stock2']
    beta = pair['beta']
    spread_mean = pair['spread_mean']
    spread_std = pair['spread_std']
    forced = pair.get('forced', False)

    spread = price1 - beta * price2
    z_score = (spread - spread_mean) / spread_std if spread_std != 0 else np.nan
    if np.isnan(z_score):
        return None

    per_trade_notional = equity * risk_per_trade_pct
    quantity1 = max(1, int(per_trade_notional / price1))
    quantity2 = max(1, int(per_trade_notional / price2))

    if z_score > z_entry_threshold:
        pos_type = 'short'
    elif z_score < -z_entry_threshold:
        pos_type = 'long'
    else:
        return None

    position = {
        'entry_time': timestamp,
        'stock1': stock1,
        'stock2': stock2,
        'type': pos_type,
        'quantity1': quantity1,
        'quantity2': quantity2,
        'entry_price1': float(price1),
        'entry_price2': float(price2),
        'beta': beta,
        'spread_mean': spread_mean,
        'spread_std': spread_std,
        'entry_spread': spread,
        'entry_z': z_score,
        'holding_period': 0,
        'entry_fees1': quantity1 * fee_per_share,
        'entry_fees2': quantity2 * fee_per_share,
        'forced': forced,
    }
    position_list.append(position)
    return position

def close_position(position, timestamp, price1, price2, exit_reason):
    qty1 = position['quantity1']
    qty2 = position['quantity2']
    entry_price1 = position['entry_price1']
    entry_price2 = position['entry_price2']

    if position['type'] == 'long':
        profit1 = (price1 - entry_price1) * qty1
        profit2 = (entry_price2 - price2) * qty2
    else:
        profit1 = (entry_price1 - price1) * qty1
        profit2 = (price2 - entry_price2) * qty2

    exit_fees1 = qty1 * fee_per_share
    exit_fees2 = qty2 * fee_per_share

    net_profit = (
        profit1 + profit2
        - position['entry_fees1'] - position['entry_fees2']
        - exit_fees1 - exit_fees2
    )

    trade_record = {
        'Entry Time': position['entry_time'],
        'Exit Time': timestamp,
        'Position': position['type'].capitalize(),
        'Stock1': position['stock1'],
        'Stock2': position['stock2'],
        'Quantity1': qty1,
        'Quantity2': qty2,
        'Entry Price1': round(entry_price1, 6),
        'Entry Price2': round(entry_price2, 6),
        'Exit Price1': round(price1, 6),
        'Exit Price2': round(price2, 6),
        'Profit1': round(profit1, 2),
        'Profit2': round(profit2, 2),
        'Fees1': round(position['entry_fees1'] + exit_fees1, 2),
        'Fees2': round(position['entry_fees2'] + exit_fees2, 2),
        'Net Profit': round(net_profit, 2),
        'Exit Reason': exit_reason,
        'Filter Status': 'FILTER_BYPASSED' if position.get('forced') else 'VALIDATED'
    }
    return trade_record, net_profit

def run_backtest(prices, timeframe='1d'):
    formation_bars = FORMATION_BARS[timeframe]
    trading_bars = TRADING_BARS[timeframe]

    min_total_bars = formation_bars + trading_bars + 20
    if len(prices) < min_total_bars:
        raise ValueError(
            f"Not enough bars. Need at least {min_total_bars}, got {len(prices)}."
        )

    all_trades = []
    equity_curve = []
    open_positions = []
    used_pairs = set()
    cash_equity = initial_balance

    start_idx = formation_bars
    session_counter = 0

    while start_idx + trading_bars < len(prices):
        session_counter += 1

        formation_slice = prices.iloc[start_idx - formation_bars:start_idx].copy()
        trading_slice = prices.iloc[start_idx:start_idx + trading_bars].copy()

        safe_print("=" * 80)
        safe_print(f"SESSION {session_counter}")
        safe_print(
            f"Formation window: {formation_slice.index[0]} -> {formation_slice.index[-1]}"
        )
        safe_print(
            f"Trading window:   {trading_slice.index[0]} -> {trading_slice.index[-1]}"
        )

        selected_pairs = analyze_pairs(formation_slice, num_pairs=num_pairs)

        n_forced = sum(1 for p in selected_pairs if p.get('forced'))
        n_validated = len(selected_pairs) - n_forced
        safe_print(
            f"Pair Selection Summary | Total={len(selected_pairs)} | "
            f"Validated={n_validated} | Forced={n_forced}"
        )

        open_positions = []
        used_pairs = set()

        for timestamp, row in trading_slice.iterrows():
            # 1) update existing positions
            still_open = []
            for pos in open_positions:
                s1 = pos['stock1']
                s2 = pos['stock2']

                if pd.isna(row.get(s1)) or pd.isna(row.get(s2)):
                    still_open.append(pos)
                    continue

                price1 = float(row[s1])
                price2 = float(row[s2])

                beta = pos['beta']
                spread = price1 - beta * price2
                spread_mean = pos['spread_mean']
                spread_std = pos['spread_std']
                z_score = (spread - spread_mean) / spread_std if spread_std != 0 else np.nan

                pos['holding_period'] += 1
                holding_period = pos['holding_period']

                entry_spread = pos['entry_spread']
                spread_pct_change = abs((spread - entry_spread) / entry_spread) if entry_spread != 0 else 0

                exit_signal = False
                exit_reason = ''

                if pos['type'] == 'long' and z_score >= -z_exit_threshold:
                    exit_signal = True
                    exit_reason = 'Exit Signal'
                elif pos['type'] == 'short' and z_score <= z_exit_threshold:
                    exit_signal = True
                    exit_reason = 'Exit Signal'
                elif holding_period >= max_holding_period_bars:
                    exit_signal = True
                    exit_reason = 'Max Holding Period Reached'
                elif spread_pct_change > stop_loss_threshold:
                    exit_signal = True
                    exit_reason = 'Stop Loss Hit'

                if exit_signal:
                    trade_record, pnl = close_position(pos, timestamp, price1, price2, exit_reason)
                    all_trades.append(trade_record)
                    cash_equity += pnl
                    pair_key = tuple(sorted([s1, s2]))
                    if pair_key in used_pairs:
                        used_pairs.remove(pair_key)
                else:
                    still_open.append(pos)

            open_positions = still_open

            # 2) open new positions if signal occurs
            for pair in selected_pairs:
                s1 = pair['stock1']
                s2 = pair['stock2']
                pair_key = tuple(sorted([s1, s2]))
                if pair_key in used_pairs:
                    continue

                if pd.isna(row.get(s1)) or pd.isna(row.get(s2)):
                    continue

                price1 = float(row[s1])
                price2 = float(row[s2])

                pos = open_position(open_positions, timestamp, pair, price1, price2, cash_equity)
                if pos is not None:
                    used_pairs.add(pair_key)
                    safe_print(
                        f"OPEN {pos['type'].upper()} | {s1}/{s2} | "
                        f"z={pos['entry_z']:.2f} | forced={pos.get('forced', False)}"
                    )

            equity_curve.append({
                'Time': timestamp,
                'Equity': cash_equity,
                'Open Positions': len(open_positions)
            })

        # Close remaining positions at end of session
        if len(open_positions) > 0:
            final_row = trading_slice.iloc[-1]
            final_ts = trading_slice.index[-1]
            for pos in open_positions:
                s1 = pos['stock1']
                s2 = pos['stock2']
                if pd.isna(final_row.get(s1)) or pd.isna(final_row.get(s2)):
                    continue
                price1 = float(final_row[s1])
                price2 = float(final_row[s2])
                trade_record, pnl = close_position(pos, final_ts, price1, price2, 'Strategy Exit')
                all_trades.append(trade_record)
                cash_equity += pnl
            open_positions = []

        start_idx += trading_bars

    trades_df = pd.DataFrame(all_trades)
    equity_df = pd.DataFrame(equity_curve)

    return trades_df, equity_df

# =============================================================================
# PERFORMANCE REPORT
# =============================================================================

def compute_metrics(trades_df, equity_df, starting_equity=100000):
    if trades_df.empty:
        return {
            'Total Trades': 0,
            'Winning Trades': 0,
            'Losing Trades': 0,
            'Win Rate %': 0.0,
            'Total Net Profit': 0.0,
            'Average Trade Profit': 0.0,
            'Final Equity': starting_equity,
            'Return %': 0.0,
            'Max Drawdown %': 0.0,
            'Profit Factor': 0.0
        }

    total_trades = len(trades_df)
    winning_trades = (trades_df['Net Profit'] > 0).sum()
    losing_trades = (trades_df['Net Profit'] < 0).sum()
    win_rate = (winning_trades / total_trades) * 100 if total_trades > 0 else 0
    total_net_profit = trades_df['Net Profit'].sum()
    avg_trade_profit = trades_df['Net Profit'].mean()
    final_equity = starting_equity + total_net_profit
    total_return_pct = (total_net_profit / starting_equity) * 100

    gross_profit = trades_df.loc[trades_df['Net Profit'] > 0, 'Net Profit'].sum()
    gross_loss = abs(trades_df.loc[trades_df['Net Profit'] < 0, 'Net Profit'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else np.inf

    max_drawdown_pct = 0.0
    if not equity_df.empty:
        eq = equity_df['Equity'].values
        running_max = np.maximum.accumulate(eq)
        drawdown = (eq - running_max) / running_max
        max_drawdown_pct = abs(drawdown.min()) * 100 if len(drawdown) > 0 else 0.0

    return {
        'Total Trades': int(total_trades),
        'Winning Trades': int(winning_trades),
        'Losing Trades': int(losing_trades),
        'Win Rate %': round(win_rate, 2),
        'Total Net Profit': round(total_net_profit, 2),
        'Average Trade Profit': round(avg_trade_profit, 2),
        'Final Equity': round(final_equity, 2),
        'Return %': round(total_return_pct, 2),
        'Max Drawdown %': round(max_drawdown_pct, 2),
        'Profit Factor': round(profit_factor, 2) if np.isfinite(profit_factor) else np.inf
    }

# =============================================================================
# MAIN
# =============================================================================

def main():
    # -------------------------------------------------------------------------
    # CHANGE THESE IF YOU WANT
    # -------------------------------------------------------------------------
    timeframe = '1d'              # choose from: '1m','5m','15m','30m','60m','1d'
    symbols = EQUITY_SYMBOLS[:]   # you can replace with your own list
    include_extra_assets = False  # set True to try FX / gold Yahoo symbols too
    # -------------------------------------------------------------------------

    safe_print("Starting backtest...")
    safe_print(f"Timeframe selected: {timeframe}")

    prices = download_price_data(symbols, timeframe=timeframe, include_extra=include_extra_assets)

    safe_print(f"Price data shape: {prices.shape}")
    safe_print(f"Date range: {prices.index.min()} -> {prices.index.max()}")

    trades_df, equity_df = run_backtest(prices, timeframe=timeframe)
    metrics = compute_metrics(trades_df, equity_df, starting_equity=initial_balance)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = f"BacktestSession_{timestamp}"
    os.makedirs(output_dir, exist_ok=True)

    trades_file = os.path.join(output_dir, "trade_history.csv")
    equity_file = os.path.join(output_dir, "equity_curve.csv")
    metrics_file = os.path.join(output_dir, "summary_metrics.csv")

    trades_df.to_csv(trades_file, index=False)
    equity_df.to_csv(equity_file, index=False)
    pd.DataFrame([metrics]).to_csv(metrics_file, index=False)

    safe_print("=" * 80)
    safe_print("BACKTEST COMPLETE")
    for k, v in metrics.items():
        safe_print(f"{k}: {v}")
    safe_print("=" * 80)

    if not trades_df.empty:
        safe_print("\nSample trades:")
        print(trades_df.head(10).to_string(index=False))
    else:
        safe_print("No trades were generated.")

    safe_print(f"\nSaved files:")
    safe_print(trades_file)
    safe_print(equity_file)
    safe_print(metrics_file)

if __name__ == "__main__":
    main()

[2026-04-10 03:47:23] Starting backtest...
[2026-04-10 03:47:23] Timeframe selected: 1d
[2026-04-10 03:47:23] Downloading data: interval=1d, period=5y
[2026-04-10 03:47:32] Downloaded close prices for 49 symbols.
[2026-04-10 03:47:32] Price data shape: (1255, 49)
[2026-04-10 03:47:32] Date range: 2021-04-11 20:00:00-04:00 -> 2026-04-08 20:00:00-04:00
[2026-04-10 03:47:32] ================================================================================
[2026-04-10 03:47:32] SESSION 1
[2026-04-10 03:47:32] Formation window: 2021-04-11 20:00:00-04:00 -> 2022-04-06 20:00:00-04:00
[2026-04-10 03:47:32] Trading window:   2022-04-07 20:00:00-04:00 -> 2022-07-10 20:00:00-04:00
[2026-04-10 03:47:32] Analyzing 1176 candidate pairs from 49 symbols.
[2026-04-10 03:48:06] Full-filter selected 20 pairs.
[2026-04-10 03:48:06] GOOGL/TMO | EG_p=0.0062964899318953185 | ADF_p=0.041572537035146004 | HL=7.1 | OOS_SR=0.322
[2026-04-10 03:48:06] MSFT/TMO | EG_p=0.0005440082505564228 | ADF_p=0.011053686410816